In [1]:
import chromaviz
from backend.RagCore.Utils.pathProvider import PathProvider
import chromadb

In [2]:
from chromaviz import visualize_collection
path_provider = PathProvider()
chroma_path = path_provider.chroma()
client = chromadb.PersistentClient(path=str(chroma_path))
collection = client.get_or_create_collection(name="cs1_1881-01-11")
#visualize_collection(collection)

In [3]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-mpnet-base-v2")

/Users/mtis/Local/Code/GitRepos/LangAI/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# 📏 Cellule 3 — Test rapide de dimension
import numpy as np

probe = embedder.encode(["dim test"])
print("Embedding shape :", probe.shape)  
from chromadb.api.types import IncludeEnum

# 1) On prend un identifiant déjà présent dans la collection
sample_id = collection.peek(limit=1)["ids"][0]

# 2) On récupère cet élément en demandant à Chroma
#    d'inclure son embedding dans la réponse
vec = collection.get(
        ids=[sample_id],
        include=[IncludeEnum.embeddings]   # ← clé pour dire « renvoie-moi le vecteur »
)["embeddings"][0]                         # ← on extrait le 1er (et unique) vecteur
print(vec.shape)

Embedding shape : (1, 768)
(768,)


In [8]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("antoinelouis/french-gte-multilingual-base", trust_remote_code=True)


query="""Selon le texte, combien de colons étaient concernés par les indemnités réclamées ?"""

query_emb = model.encode([query],normalize_embeddings=True)             



out = collection.query(
        query_embeddings=query_emb,
        n_results=10,
        include=["documents", "distances"]
)
print("Metadata =", collection.metadata)  
# 4) Parcourez le résultat
for rank, (doc, dist) in enumerate(
        zip(out["documents"][0], out["distances"][0]), 1):
    print(f"{rank}. distance={dist:.4f} → {doc[:200]} …")

Number of requested results 10 is greater than number of elements in index 6, updating n_results = 6


Metadata = {'hnsw:space': 'cosine'}
1. distance=0.2091 → 

HESOLUTION de la 17* commission des pti.

tions, insérée dans le feuilleton du 2ddeembra1880, devenue définitive aux termes de l'article 66 du règlement.

aArt. 66. — Tout député, dans le mois de la …
2. distance=0.4176 → EXCUSES ET DEMANDES DE CONGÉS MM. Dautresme, Bourgeois, Labuze,Hémon,Monteils et Péronne, s'excusent de ne pouvoir assister aux premières séances de la Chambre.

MM. Descamps, Harispe, Fauré (Gers), D …
3. distance=0.4796 → 

PBÉSIDBNCB DE M. DESSBAUX, DOYEN D'AGE.

La séance est ouverte à deux heures dixminuttS.

MDreyfas, l'un des secrétaires provisoires, donne lecture du procès-verbal de la séance du11janvier.

- SI.  …
4. distance=0.4802 → .

CHAMBRE DES DÉPUTÉS Session ordinaire de 1881

COIPTE RENDU IN EXTENSO. - 28 SÉANCE Séance du 20 janvier 1881 SOMMAIRE Incident : M. Cuneo d'Ornano.

Communication, par M. le président, d'une lettr …
5. distance=0.4943 → 

ALLOCUTION DE M. LE PRÉSIDENT D'AGE

M. le pr

In [6]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")  # ou le modèle de ton choix

s1 = """des lois sérieuses et qui ne prêtent pas aux justes reproches que nous adressons à la loi da juillet 1873.\n\nPlacez-vous en face de ces difficultés.Hest:rfacile de dire : 
Il ne s'agit que d'une prise ea considération; votons-la d'abord, nous verrons après ce que nous pourrons en faire.\n\nJe réponds que vous aurez ainsi 
produit un effet qui tournera contre le but que vous pours ui* vez, si vous êtes obligés, plus tard, de reconnaître que vous vous 
heurtez à une impossi\" b lité matérielle; à moins que vous n'ayezla\n\npensée d'employer les millions de l'Etat à\n\nl'usage que """


s2 = """des lois sérieuses et qui ne prêtent pas aux justes reproches que nous adressons à la loi da juillet 1873.\n\nPlacez-vous en face de ces difficultés.Hest:rfacile de dire : 
Il ne s'agit que d'une prise ea considération; votons-la d'abord, nous verrons après ce que nous pourrons en faire.\n\nJe réponds que vous aurez ainsi
produit un effet qui tournera contre le but que vous pours ui* vez, si vous êtes obligés, plus tard, de reconnaître que vous vous 
heurtez à une impossi\" b lité matérielle; à moins que vous n'ayezla\n\npensée d'employer les millions de l'Etat à\n\nl'usage que 
je vous ai dit; ce qui serait, je le crains, difficilement approuvé par le paysTelles sont, messienrs, les observations queIdGouvernement
avait la devoir de vous soumettre, et qui l'ont déterminé à vous propo* ser de ne pas prononcer la prise en coasidddération. 
(Applaudissements au centre et sur divers bancs à giucha et à droite.) M. le président. La parole est à M.lerapporteur."""


emb1 = model.encode(s1, convert_to_tensor=True, normalize_embeddings=True)
emb2 = model.encode(s2, convert_to_tensor=True, normalize_embeddings=True)

# 2) Similarité cosinus
sim = util.cos_sim(emb1, emb2)          # shape (1,1)
print(float(sim))                       # ex. 0.81  → plus proche de 1 = phrases très proches

1.0
